# Fatores Socioeconômicos no Desempenho do ENEM × Indicadores Estaduais do IBGE
**Entrega 1 — Análise Exploratória, ETL e Dashboard de KPIs**

**Objetivo:** unir uma amostra de microdados do ENEM a indicadores estaduais do IBGE para (i) gerar estatísticas descritivas, (ii) calcular 20 KPIs e (iii) construir visualizações que sustentem o storytelling do projeto.

> ⚠️ **Nota metodológica:** os dados deste notebook são **simulados** (semente fixa, reprodutíveis), construídos para reproduzir relações plausíveis (renda e escola privada ↑ nota; IDH ↑ nota). Os valores do IBGE são **aproximados e ilustrativos**. Para a entrega final, basta substituir as funções da Seção 2 por `pd.read_csv()` dos dados reais (INEP/IBGE) mantendo os mesmos nomes de colunas.

## 1. Configuração e importações

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

SEED = 42
rng = np.random.default_rng(SEED)

# Estilo global dos gráficos
sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 13})

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 30)

## 2. Simulação / carregamento dos dados
Duas funções geradoras:
- `gerar_ibge()` → tabela estadual (27 UFs) com `UF`, `REGIAO`, `IDH_ESTADUAL`, `RENDA_PER_CAPITA` (R$/mês) e `POPULACAO`.
- `gerar_enem()` → amostra de inscritos. A nota de cada aluno depende de uma "habilidade latente" que combina **renda familiar**, **tipo de escola** e **IDH da UF**, mais ruído individual. Alunos ausentes ficam com notas `NaN` (como nos microdados reais do INEP).

`TP_PRESENCA`: 0 = faltou · 1 = presente · 2 = eliminado.

In [ ]:
def gerar_ibge() -> pd.DataFrame:
    """Indicadores estaduais (valores aproximados, uso didático)."""
    dados = [
        # UF, Região, IDH, Renda per capita (R$), População
        ("RO", "Norte", 0.725, 1600, 1_581_196), ("AC", "Norte", 0.710, 1300, 830_018),
        ("AM", "Norte", 0.700, 1200, 3_941_613), ("RR", "Norte", 0.707, 1500, 636_707),
        ("PA", "Norte", 0.690, 1100, 8_120_131), ("AP", "Norte", 0.688, 1300, 733_759),
        ("TO", "Norte", 0.731, 1500, 1_511_460),
        ("MA", "Nordeste", 0.676, 900, 6_776_699), ("PI", "Nordeste", 0.690, 1000, 3_271_199),
        ("CE", "Nordeste", 0.734, 1100, 8_794_957), ("RN", "Nordeste", 0.728, 1400, 3_302_729),
        ("PB", "Nordeste", 0.698, 1200, 3_974_687), ("PE", "Nordeste", 0.719, 1200, 9_058_931),
        ("AL", "Nordeste", 0.684, 1000, 3_127_683), ("SE", "Nordeste", 0.702, 1300, 2_210_004),
        ("BA", "Nordeste", 0.714, 1100, 14_141_626),
        ("MG", "Sudeste", 0.774, 1900, 20_538_718), ("ES", "Sudeste", 0.771, 1900, 3_833_712),
        ("RJ", "Sudeste", 0.762, 2300, 16_055_174), ("SP", "Sudeste", 0.806, 2400, 44_411_238),
        ("PR", "Sul", 0.769, 2000, 11_444_380), ("SC", "Sul", 0.792, 2500, 7_610_361),
        ("RS", "Sul", 0.771, 2300, 10_882_965),
        ("MS", "Centro-Oeste", 0.742, 2000, 2_757_013), ("MT", "Centro-Oeste", 0.736, 1900, 3_658_649),
        ("GO", "Centro-Oeste", 0.737, 1900, 7_056_495), ("DF", "Centro-Oeste", 0.814, 3500, 2_817_381),
    ]
    return pd.DataFrame(dados, columns=["UF", "REGIAO", "IDH_ESTADUAL", "RENDA_PER_CAPITA", "POPULACAO"])


def gerar_enem(df_ibge: pd.DataFrame, n: int = 3000, rng=rng) -> pd.DataFrame:
    """Amostra fictícia, mas estatisticamente coerente, de inscritos no ENEM."""
    ibge = df_ibge.set_index("UF")

    # UF sorteada proporcionalmente à população (suavizada)
    pesos = ibge["POPULACAO"] ** 0.8
    uf = rng.choice(ibge.index, size=n, p=(pesos / pesos.sum()).values)

    idh = ibge.loc[uf, "IDH_ESTADUAL"].values
    renda_pc = ibge.loc[uf, "RENDA_PER_CAPITA"].values
    renda_norm = (renda_pc - renda_pc.min()) / (renda_pc.max() - renda_pc.min())

    # Escola privada: mais provável em UFs com IDH maior (~10% a ~35%)
    p_priv = np.clip(0.10 + (idh - 0.67) * 1.8, 0.05, 0.45)
    privada = rng.random(n) < p_priv
    tp_escola = np.where(privada, "Privada", "Pública")

    # Q006 (A..Q): 17 faixas de renda familiar; depende de UF e escola
    idx_renda = np.clip(np.round(rng.normal(2.0 + 3.0 * renda_norm + 4.5 * privada, 2.0)), 0, 16).astype(int)
    q006 = np.array([chr(65 + i) for i in idx_renda])

    # Habilidade latente (padronizada): renda + escola + IDH + ruído individual
    z_renda = (idx_renda - idx_renda.mean()) / idx_renda.std()
    z_idh = (idh - idh.mean()) / idh.std()
    habilidade = 0.40 * z_renda + 0.30 * privada + 0.15 * z_idh + rng.normal(0, 0.80, n)

    def nota(media, sd, peso):
        return np.clip(media + peso * habilidade + rng.normal(0, sd, n), 0, 1000)

    cn, ch, lc, mt = nota(485, 55, 55), nota(520, 60, 55), nota(510, 50, 45), nota(530, 70, 75)
    red = np.clip(np.round((570 + 110 * habilidade + rng.normal(0, 110, n)) / 20) * 20, 0, 1000)
    red[rng.random(n) < 0.015] = 0  # redações anuladas / zeradas

    # Presença: ausência é mais frequente em rendas baixas
    p_falta = np.clip(0.30 - 0.012 * idx_renda - 0.05 * privada, 0.05, 0.45)
    sorteio = rng.random(n)
    presenca = np.where(sorteio < p_falta, 0, np.where(sorteio > 0.985, 2, 1))

    df = pd.DataFrame({
        "UF": uf, "TP_ESCOLA": tp_escola, "Q006": q006,
        "NU_NOTA_CN": cn.round(1), "NU_NOTA_CH": ch.round(1), "NU_NOTA_LC": lc.round(1),
        "NU_NOTA_MT": mt.round(1), "NU_NOTA_REDACAO": red, "TP_PRESENCA": presenca,
    })

    # Quem faltou ou foi eliminado não tem notas (NaN), como nos microdados reais
    cols_nota = [c for c in df.columns if c.startswith("NU_NOTA")]
    df.loc[df["TP_PRESENCA"] != 1, cols_nota] = np.nan

    # Pequena parcela de nulos aleatórios (falhas de registro) para exercitar o tratamento
    presentes = df.index[df["TP_PRESENCA"] == 1]
    falhas = rng.choice(presentes, size=int(0.01 * len(presentes)), replace=False)
    df.loc[falhas, "NU_NOTA_CN"] = np.nan
    return df


df_ibge = gerar_ibge()
df_enem = gerar_enem(df_ibge, n=3000)

print(f"ENEM: {df_enem.shape[0]:,} linhas × {df_enem.shape[1]} colunas | IBGE: {df_ibge.shape}")
df_enem.head()

## 3. ETL e Engenharia de Recursos
Etapas: **(1)** diagnóstico de nulos → **(2)** merge com IBGE (`m:1`, validado) → **(3)** `NOTA_FINAL` → **(4)** faixas de renda → **(5)** tratamento de nulos.

**Decisão de tratamento:** nulos nas notas de quem faltou/foi eliminado são *estruturais* (não há prova), então **não são imputados**: a análise de desempenho usa apenas **presentes com as 5 notas válidas**, enquanto indicadores de abstenção usam a base completa. Nulos "acidentais" em presentes (falha de registro) são removidos do recorte de desempenho.

Mapeamento de `Q006`: **A–D** (até 2 SM) → Baixa Renda · **E–H** (2 a 5 SM) → Média Renda · **I–Q** (> 5 SM) → Alta Renda.

In [ ]:
COLS_NOTAS = ["NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT", "NU_NOTA_REDACAO"]

def mapear_faixa_renda(q006: str) -> str:
    if q006 in list("ABCD"):
        return "Baixa Renda"
    if q006 in list("EFGH"):
        return "Média Renda"
    return "Alta Renda"


def etl(df_enem: pd.DataFrame, df_ibge: pd.DataFrame) -> pd.DataFrame:
    # 1) Diagnóstico de nulos
    print("Nulos por coluna (antes do tratamento):")
    print(df_enem.isna().sum()[lambda s: s > 0].to_string(), "\n")

    # 2) Merge com validação de cardinalidade (muitos alunos -> uma UF)
    df = df_enem.merge(df_ibge, on="UF", how="left", validate="m:1")
    assert df["IDH_ESTADUAL"].notna().all(), "Há UFs sem correspondência no IBGE!"

    # 3) Nota final = média das 5 provas (NaN se faltar alguma)
    df["NOTA_FINAL"] = df[COLS_NOTAS].mean(axis=1, skipna=False)

    # 4) Faixa de renda categórica ordenada + índice numérico (A=0 ... Q=16)
    ordem = ["Baixa Renda", "Média Renda", "Alta Renda"]
    df["FAIXA_RENDA"] = pd.Categorical(df["Q006"].map(mapear_faixa_renda), categories=ordem, ordered=True)
    df["RENDA_IDX"] = df["Q006"].map(lambda x: ord(x) - 65)

    # 5) Flags de presença e padronização de rótulos
    df["PRESENTE"] = (df["TP_PRESENCA"] == 1)
    df["FALTOU"] = (df["TP_PRESENCA"] == 0)
    return df


df = etl(df_enem, df_ibge)

# Recorte analítico: presentes com NOTA_FINAL válida
df_pres = df[df["PRESENTE"] & df["NOTA_FINAL"].notna()].copy()

print(f"Base completa: {len(df):,} | Presentes com notas válidas: {len(df_pres):,} "
      f"({len(df_pres)/len(df):.1%})")
df_pres[["UF", "TP_ESCOLA", "FAIXA_RENDA", "NOTA_FINAL", "IDH_ESTADUAL"]].head()

## 4. Estatísticas descritivas
Visão geral das notas e das variáveis categóricas antes de calcular os KPIs.

In [ ]:
print("=== Estatísticas descritivas das notas (presentes válidos) ===")
display(df_pres[COLS_NOTAS + ["NOTA_FINAL"]].describe().T)

print("\n=== Distribuição de variáveis categóricas (base completa) ===")
for col in ["TP_ESCOLA", "FAIXA_RENDA", "TP_PRESENCA"]:
    display(df[col].value_counts(normalize=True).mul(100).round(1).rename("%").to_frame())

print("\n=== Nota final média por tipo de escola e faixa de renda ===")
display(df_pres.pivot_table(index="FAIXA_RENDA", columns="TP_ESCOLA", values="NOTA_FINAL",
                            aggfunc=["mean", "count"], observed=True))

## 5. Cálculo dos 20 KPIs
Os KPIs estão organizados em **4 eixos** (5 por eixo). Cada um é registrado via função auxiliar `kpi()` em uma tabela única, com valor numérico, unidade e uma observação de contexto (ex.: qual UF). Tabelas intermediárias por UF ficam guardadas para reutilização nos gráficos.

**Definições adotadas:**
- *Exatas* = média de Ciências da Natureza e Matemática por aluno.
- *Abstenção* = `TP_PRESENCA == 0` sobre a base completa; *Presença* = `TP_PRESENCA == 1`.
- *Índice de Desigualdade Regional* = coeficiente de variação (%) das médias estaduais da nota final.
- *Gap de renda entre UFs* = nota média da UF de maior renda per capita − nota média da UF de menor renda per capita.

In [ ]:
registro = []

def kpi(eixo, nome, valor, unidade="", obs=""):
    registro.append({"Eixo": eixo, "KPI": nome, "Valor": valor, "Unidade": unidade, "Observação": obs})

# ---------- Tabelas auxiliares por UF ----------
media_uf = df_pres.groupby("UF")["NOTA_FINAL"].mean().rename("NOTA_MEDIA")
tab_uf = (df_ibge.set_index("UF").join(media_uf)
          .join(df.groupby("UF").size().rename("INSCRITOS"))
          .join(df.groupby("UF")["RENDA_IDX"].mean().rename("RENDA_IDX_MEDIA")))
tab_uf["DENSIDADE_100K"] = tab_uf["INSCRITOS"] / tab_uf["POPULACAO"] * 100_000

A, B, C, D = "A) Desempenho Acadêmico", "B) Perfil Socioeconômico", "C) Cruzamento IBGE", "D) Operacional / Resumo"

# ---------------- A) Desempenho Acadêmico ----------------
kpi(A, "Média Geral por UF (melhor UF)", media_uf.max(), "pts", f"{media_uf.idxmax()} | pior: {media_uf.idxmin()} ({media_uf.min():.1f})")
kpi(A, "Mediana da Redação", df_pres["NU_NOTA_REDACAO"].median(), "pts")
exatas = (df_pres["NU_NOTA_CN"] + df_pres["NU_NOTA_MT"]) / 2
kpi(A, "Desvio-padrão de Exatas (CN+MT)", exatas.std(), "pts")
kpi(A, "Taxa de Notas > 700 (Nota Final)", (df_pres["NOTA_FINAL"] > 700).mean() * 100, "%")
kpi(A, "Taxa de Zeros na Redação", (df_pres["NU_NOTA_REDACAO"] == 0).mean() * 100, "%")

# ---------------- B) Perfil Socioeconômico ----------------
renda_uf = tab_uf["RENDA_IDX_MEDIA"]
kpi(B, "Média de Renda por UF (índice Q006, maior)", renda_uf.max(), "índice 0–16", f"{renda_uf.idxmax()} | menor: {renda_uf.idxmin()} ({renda_uf.min():.2f})")
pct_escola = df["TP_ESCOLA"].value_counts(normalize=True) * 100
kpi(B, "% Escola Pública", pct_escola["Pública"], "%", f"Privada: {pct_escola['Privada']:.1f}%")
med_escola = df_pres.groupby("TP_ESCOLA")["NOTA_FINAL"].mean()
kpi(B, "Razão de Notas Privada/Pública", med_escola["Privada"] / med_escola["Pública"], "razão",
    f"Privada {med_escola['Privada']:.1f} vs Pública {med_escola['Pública']:.1f}")
dist_renda = df["FAIXA_RENDA"].value_counts(normalize=True).mul(100)
kpi(B, "Distribuição por Faixa de Renda (% Baixa Renda)", dist_renda["Baixa Renda"], "%",
    f"Média: {dist_renda['Média Renda']:.1f}% | Alta: {dist_renda['Alta Renda']:.1f}%")
kpi(B, "% Abstenção", df["FALTOU"].mean() * 100, "%")

# ---------------- C) Cruzamento IBGE ----------------
corr_idh = tab_uf["IDH_ESTADUAL"].corr(tab_uf["NOTA_MEDIA"])
kpi(C, "Correlação IDH × Nota Média (Pearson, nível UF)", corr_idh, "r", f"n = {tab_uf['NOTA_MEDIA'].notna().sum()} UFs")
uf_rico, uf_pobre = tab_uf["RENDA_PER_CAPITA"].idxmax(), tab_uf["RENDA_PER_CAPITA"].idxmin()
kpi(C, "Gap de Nota: UF de maior vs menor renda per capita", tab_uf.loc[uf_rico, "NOTA_MEDIA"] - tab_uf.loc[uf_pobre, "NOTA_MEDIA"],
    "pts", f"{uf_rico} − {uf_pobre}")
dens_nac = len(df) / df_ibge["POPULACAO"].sum() * 100_000
kpi(C, "Densidade de Inscritos / 100 mil hab. (amostra)", dens_nac, "insc./100k",
    f"maior UF: {tab_uf['DENSIDADE_100K'].idxmax()} ({tab_uf['DENSIDADE_100K'].max():.2f})")
abst_renda = df.groupby("FAIXA_RENDA", observed=True)["FALTOU"].mean() * 100
kpi(C, "Taxa de Abstenção por Renda (Baixa − Alta)", abst_renda["Baixa Renda"] - abst_renda["Alta Renda"], "p.p.",
    f"Baixa {abst_renda['Baixa Renda']:.1f}% | Média {abst_renda['Média Renda']:.1f}% | Alta {abst_renda['Alta Renda']:.1f}%")
kpi(C, "Índice de Desigualdade Regional (CV das médias por UF)", media_uf.std() / media_uf.mean() * 100, "%")

# ---------------- D) Operacional / Resumo ----------------
kpi(D, "Total de Inscritos na Amostra", len(df), "alunos")
kpi(D, "Média Geral de Presença", df["PRESENTE"].mean() * 100, "%")
baixa = df_pres[df_pres["FAIXA_RENDA"] == "Baixa Renda"]
kpi(D, "% Alunos Baixa Renda com Nota > 600", (baixa["NOTA_FINAL"] > 600).mean() * 100, "%", f"base: {len(baixa):,} alunos")
kpi(D, "Maior Nota Geral", df_pres["NOTA_FINAL"].max(), "pts")
kpi(D, "Menor Nota Geral", df_pres["NOTA_FINAL"].min(), "pts")

df_kpis = pd.DataFrame(registro)
df_kpis.insert(0, "#", range(1, len(df_kpis) + 1))
assert len(df_kpis) == 20, f"Esperados 20 KPIs, encontrados {len(df_kpis)}"

# Exibição organizada por eixo
for eixo, grupo in df_kpis.groupby("Eixo", sort=True):
    print(f"\n{eixo}")
    display(grupo.drop(columns="Eixo").set_index("#").style.format({"Valor": "{:,.2f}"}))

## 6. Painel de visualizações (dashboard analítico)
Cinco gráficos, cada um respondendo a uma pergunta do storytelling:
1. Quais UFs têm melhor desempenho médio?
2. Escola pública × privada: diferença de nível e de dispersão?
3. O desenvolvimento humano do estado explica a nota?
4. Como se distribuem as notas de redação?
5. Como a composição de renda varia entre as regiões?

In [ ]:
COR_PRINCIPAL = "#2F5D8A"

# ---------- Gráfico 1: Média da Nota Final por UF ----------
ordem_uf = media_uf.sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(x=ordem_uf.index, y=ordem_uf.values, color=COR_PRINCIPAL, ax=ax)
ax.axhline(df_pres["NOTA_FINAL"].mean(), color="crimson", ls="--", lw=1.5,
           label=f"Média nacional da amostra ({df_pres['NOTA_FINAL'].mean():.0f})")
ax.set(title="Gráfico 1 — Média da Nota Final por UF", xlabel="UF", ylabel="Nota final média",
       ylim=(max(0, ordem_uf.min() - 40), ordem_uf.max() + 20))
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ---------- Gráfico 2: Notas por tipo de escola (Boxplot) ----------
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df_pres, x="TP_ESCOLA", y="NOTA_FINAL", hue="TP_ESCOLA", order=["Pública", "Privada"],
            hue_order=["Pública", "Privada"], palette=["#C0504D", COR_PRINCIPAL], legend=False,
            flierprops={"marker": "o", "markersize": 3, "alpha": 0.5}, ax=ax)
for i, tp in enumerate(["Pública", "Privada"]):
    ax.text(i, df_pres["NOTA_FINAL"].max() * 0.99, f"média = {med_escola[tp]:.0f}", ha="center", fontsize=10)
ax.set(title="Gráfico 2 — Nota Final por Tipo de Escola", xlabel="Tipo de escola", ylabel="Nota final")
plt.tight_layout(); plt.show()

In [ ]:
# ---------- Gráfico 3: IDH × Nota média (scatter + tendência) ----------
dados3 = tab_uf.dropna(subset=["NOTA_MEDIA"]).reset_index()
fig, ax = plt.subplots(figsize=(9, 6))
sns.regplot(data=dados3, x="IDH_ESTADUAL", y="NOTA_MEDIA", ci=95,
            scatter_kws={"s": 60, "color": COR_PRINCIPAL, "alpha": 0.85},
            line_kws={"color": "crimson", "lw": 2}, ax=ax)
for _, r in dados3.iterrows():
    ax.annotate(r["UF"], (r["IDH_ESTADUAL"], r["NOTA_MEDIA"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set(title=f"Gráfico 3 — IDH Estadual × Nota Média no ENEM (r = {corr_idh:.2f})",
       xlabel="IDH estadual (IBGE)", ylabel="Nota final média da UF")
plt.tight_layout(); plt.show()

In [ ]:
# ---------- Gráfico 4: Distribuição das notas de Redação ----------
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=df_pres, x="NU_NOTA_REDACAO", hue="TP_ESCOLA", bins=25, kde=True, stat="density",
             common_norm=False, element="step", palette={"Pública": "#C0504D", "Privada": COR_PRINCIPAL}, ax=ax)
ax.axvline(df_pres["NU_NOTA_REDACAO"].median(), color="black", ls="--", lw=1,
           label=f"Mediana geral ({df_pres['NU_NOTA_REDACAO'].median():.0f})")
ax.set(title="Gráfico 4 — Distribuição das Notas de Redação", xlabel="Nota de redação", ylabel="Densidade")
ax.legend(title="Escola", loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
# ---------- Gráfico 5: Faixa de renda por Região (barras empilhadas, 100%) ----------
prop = (pd.crosstab(df["REGIAO"], df["FAIXA_RENDA"], normalize="index") * 100)[["Baixa Renda", "Média Renda", "Alta Renda"]]
prop = prop.sort_values("Baixa Renda", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
prop.plot(kind="bar", stacked=True, color=["#C0504D", "#E8B04B", "#2F5D8A"], width=0.75, ax=ax)
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.0f%%", label_type="center", color="white", fontsize=9)
ax.set(title="Gráfico 5 — Proporção de Alunos por Faixa de Renda e Região", xlabel="Região", ylabel="% de alunos")
ax.legend(title="Faixa de renda", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 7. Síntese para o storytelling
Use os KPIs e gráficos acima para estruturar o relatório (reveja os números após rodar, pois a simulação é reprodutível pela `SEED`):
- **Desigualdade escolar:** razão Privada/Pública (KPI B3) e Gráfico 2.
- **Território:** correlação IDH × nota (KPI C1), gap entre UFs (C2), índice de desigualdade regional (C5) e Gráficos 1 e 3.
- **Acesso:** abstenção por renda (C4) e Gráfico 5 mostram quem sequer chega à prova.
- **Mobilidade:** % de baixa renda com nota > 600 (D3) como indicador de "exceções" à regra.

**Limitações a declarar no relatório:** dados simulados; correlação ecológica (nível UF) não implica causalidade individual; amostra não é probabilística; valores IBGE aproximados.